# Lander Videos

Elise flies by feel.

This notebook records and displays videos for a high-scoring SolarSystemLander checkpoint.

## Set up

In [ ]:
# cell: package-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

In [ ]:
# cell: video-setup; requires: package-setup

from dqn.model import DQN
from hpo.evaluation.video import checkpoint_metadata, record_video
from hpo.environments.solar_system_lander.env import DEFAULT_WORLD_MIX, EnvFactory, World

_OBSERVATION_MODE = "10d"
STUDY_NAME = f"solar_system_lander_{_OBSERVATION_MODE}_elise_stp"

ENV_FACTORY = EnvFactory(_OBSERVATION_MODE, world_mix=DEFAULT_WORLD_MIX)

## Record videos

In [ ]:
# cell: video-cases; requires: video-setup

WORLDS = [
    World.MERCURY,
    World.VENUS,
    World.EARTH,
    World.MOON,
    World.MARS,
]
SEEDS = [7]  # [7, 42, 1911, 2011, 4711]

In [ ]:
# cell: prepare-colored-eagle-checkpoint; requires: video-setup

from pathlib import Path
import hashlib
import json
import sys
import types

import torch
from hpo.evaluation.video import InfraCfg

VIDEO_STUDY_NAME = f"{STUDY_NAME}_colored_video"
SOURCE_CFG = InfraCfg()
VIDEO_CFG = InfraCfg()

VIDEO_CFG.prepare()

_original_checkpoint = SOURCE_CFG.checkpoint_path(STUDY_NAME)
_colored_checkpoint = VIDEO_CFG.checkpoint_path(VIDEO_STUDY_NAME)
_metadata_path = SOURCE_CFG.checkpoint_metadata_path(STUDY_NAME)
_colored_metadata_path = VIDEO_CFG.checkpoint_metadata_path(VIDEO_STUDY_NAME)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if not _original_checkpoint.exists():
    raise FileNotFoundError(_original_checkpoint)
if not _metadata_path.exists():
    raise FileNotFoundError(_metadata_path)

print(f"Original checkpoint: {_original_checkpoint}")
print(f"Original sha256: {_sha256(_original_checkpoint)}")
print(f"Colored-video checkpoint: {_colored_checkpoint}")

if _colored_checkpoint.exists():
    print("Colored-video checkpoint already exists; leaving it unchanged.")
    print(f"Colored-video sha256: {_sha256(_colored_checkpoint)}")
else:
    _colored_checkpoint.parent.mkdir(parents=True, exist_ok=True)

    # Temporary aliases only for reading this legacy checkpoint.
    import hpo as _hpo_pkg
    import hpo.environments.solar_system_lander.env as _current_ssl_env

    _legacy_ssl = types.ModuleType("hpo.solar_system_lander")
    _legacy_ssl.__path__ = []
    for _name in dir(_current_ssl_env):
        if not _name.startswith("__"):
            setattr(_legacy_ssl, _name, getattr(_current_ssl_env, _name))
    _legacy_ssl.env = _current_ssl_env
    sys.modules.setdefault("hpo.solar_system_lander", _legacy_ssl)
    sys.modules.setdefault("hpo.solar_system_lander.env", _current_ssl_env)
    setattr(_hpo_pkg, "solar_system_lander", _legacy_ssl)

    _legacy_checkpoint = torch.load(_original_checkpoint, map_location="cpu", weights_only=False)
    _metadata = json.loads(_metadata_path.read_text(encoding="utf-8"))
    _metadata["source_study_name"] = STUDY_NAME
    _metadata["migrated_from_checkpoint"] = str(_original_checkpoint)
    _metadata["migrated_from_checkpoint_sha256"] = _sha256(_original_checkpoint)

    torch.save(
        {
            "version": int(_legacy_checkpoint.get("version", 1)),
            "model_state_dict": _legacy_checkpoint["model_state_dict"],
            "metadata": _metadata,
        },
        _colored_checkpoint,
    )
    _colored_metadata_path.write_text(json.dumps(_metadata, indent=2), encoding="utf-8")
    print("Wrote colored-video checkpoint.")
    print(f"Colored-video sha256: {_sha256(_colored_checkpoint)}")

In [ ]:
# cell: record-videos-colored-eagle; requires: video-setup, video-cases, prepare-colored-eagle-checkpoint

from hpo.evaluation.rendering.solar_system_lander import render_config

video_paths = [
    record_video(
        DQN,
        ENV_FACTORY.make_env(world, render_mode="rgb_array"),
        study_name=VIDEO_STUDY_NAME,
        seed=seed,
        # the skin argument is optional; "detailed_eagle" is also available
        render_cfg=render_config([world], overlay=True, skin="colored_eagle"),
        cfg=VIDEO_CFG,
    )
    for world in WORLDS
    for seed in SEEDS
]

## Play videos

In [ ]:
# cell: inspect-video-conditions; requires: video-setup, video-cases

from hpo.evaluation.video import display_video, show_video_conditions

show_video_conditions(ENV_FACTORY, WORLDS, SEEDS)

In [ ]:
display_video(video_paths, 1)